# Generalized ADCS — SSC26 quickstart

**Poster SSC26-P2-54 · *Generalized Attitude Control for Small Spacecraft***

Any published attitude control law drops into Stage 2 of the pipeline. The
adapter around it — goal formulation, compensation, allocation — fits the law
to whatever hardware you actually have.

Run the two cells below. No local install; nothing to configure.

- Cell 1 installs the package (~1–2 min in Colab).
- Cell 2 flies a **3 magnetorquer + 1 reaction wheel** CubeSat from 90° off
  target to under a tenth of a degree, and plots it.

In [ ]:
# 1. Install. Same line the poster prints.
!pip install -q generalized-adcs
print("installed")

In [ ]:
# 2. A converging 3MTQ + 1RW pointing run, end to end.
import numpy as np, matplotlib.pyplot as plt
import ADCS
from ADCS.pipeline import PipelineController
from ADCS.pipeline.control_law import PD_Law
from ADCS.pipeline.data import AllocationConfig
from ADCS.helpers.math_helpers import normalize, rot_mat
from ADCS.state import State

# --- the bus: 3 magnetorquers, 1 reaction wheel on +z ---
acts  = [ADCS.MTQ(axis=a, max_torque=0.2) for a in np.eye(3)]
acts += [ADCS.RW(axis=np.array([0, 0, 1.0]), max_torque=0.0023,
                 J=5.7e-6, h=0.0, h_max=0.0036)]
sens  = [ADCS.MTM(axis=a) for a in np.eye(3)]
sens += [ADCS.Gyro(axis=a) for a in np.eye(3)]
sat = ADCS.Satellite(mass=4.0, J_0=np.diag([0.03, 0.03, 0.01]),
                     actuators=acts, sensors=sens,
                     boresight=np.array([0, 0, 1.0]))

# --- Stage 2: the control law. Swap this line for your own. ---
law = PD_Law(kp=5e-4, kd=1e-1, eps=1.0)

# --- Stages 1/4/5: the adapter. 'lp' uses the wheel AND the magnetorquers. ---
ctrl = PipelineController(sat, law,
                          alloc_config=AllocationConfig(method='lp'))

# --- point the +z boresight at an inertial direction, from 90 deg away ---
goal = ADCS.goals.ECI_Goal(normalize(np.array([0.0, 0.0, 1.0])))
os0 = ADCS.Orbital_State(ephem=ADCS.Ephemeris(), J2000=0.22,
                         R=7000 * np.array([0.0, -0.7071, 0.7071]),
                         V=np.array([7.5, 0.0, 0.0]))
x0 = State(w=np.array([0.005, -0.003, 0.004]),
           q=normalize(np.array([0.5, 0.5, -0.5, 0.5])),
           h=np.zeros(1))

res = ADCS.simulate(x=x0, satellite=sat, controller=ctrl, goal=goal,
                    os0=os0, dt=1.0, tf=3000.0)

# --- pointing error: angle between the boresight and the target, in ECI ---
run = res.first()
X  = run.state_hist                   # list of State
Th = np.asarray(run.target_hist, float)
Bh = np.asarray(run.boresight_hist, float)
n  = min(len(X), len(Th), len(Bh))
err = np.full(n, np.nan)
for i in range(n):
    tgt = Th[i][1:4]                      # NaN scalar part => vector goal
    tgt = tgt / np.linalg.norm(tgt)
    b_eci = rot_mat(X[i].q) @ (Bh[i] / np.linalg.norm(Bh[i]))
    err[i] = np.degrees(np.arccos(np.clip(float(b_eci @ tgt), -1, 1)))
t = np.asarray(run.time_s, float)[:n] if run.time_s is not None else np.arange(n)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, err, color='#2a78d6', lw=2)
ax.set_yscale('log')
ax.set_xlabel('time [s]'); ax.set_ylabel('pointing error [deg]')
ax.set_title('3MTQ + 1RW, PD law through the LP allocator', loc='left')
ax.grid(True, color='#d8d7d2', lw=0.8, alpha=0.7); ax.set_axisbelow(True)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

print(f"start {err[0]:.1f} deg  ->  final {err[-1]:.3f} deg")

## Now change one thing

Each of these is a one-line edit to the cell above.

**Swap the allocator (Stage 5).** `'magnetic_cross'` is the published
magnetorquer-only behaviour — the wheel goes idle. `'qp'` recovers more torque
magnitude but lets the direction tilt when saturated; `'lp'` holds the
direction exactly and gives up magnitude instead.

```python
AllocationConfig(method='lp')   # direction kept
AllocationConfig(method='qp')   # size kept, tilts
AllocationConfig(method='qpw')  # perp error x100
AllocationConfig(method='magnetic_cross')  # MTQ only
```

**Swap the goal (Stage 1).** The same law and bus, a different objective —
Stage 1 converts each goal into whatever error signals the law asked for.

```python
goal = ADCS.goals.Nadir_Goal()
```

**Bring your own law (Stage 2).** Implement one method and declare what you
want to be handed. If your law already does its own gyroscopic compensation,
say so and the pipeline stops adding it — no double-counting.

In [ ]:
from ADCS.pipeline.control_law import ControlLaw, LawInterface

class MyLaw(ControlLaw):
    interface = LawInterface()   # full attitude + rate
    kp, kd = 5e-4, 1e-1

    def compute(self, q_err, w_err=None, **kw):
        return -(self.kp * q_err + self.kd * w_err)

mine = PipelineController(sat, MyLaw(),
                          alloc_config=AllocationConfig(method='lp'))
res2 = ADCS.simulate(x=x0, satellite=sat, controller=mine, goal=goal,
                     os0=os0, dt=1.0, tf=3000.0)

run2 = res2.first()
X2  = run2.state_hist
Th2 = np.asarray(run2.target_hist, float)
Bh2 = np.asarray(run2.boresight_hist, float)
n2  = min(len(X2), len(Th2), len(Bh2))
e2  = np.array([
    np.degrees(np.arccos(np.clip(float(
        (rot_mat(X2[i].q) @ (Bh2[i] / np.linalg.norm(Bh2[i])))
        @ (Th2[i][1:4] / np.linalg.norm(Th2[i][1:4]))), -1, 1)))
    for i in range(n2)])
print(f"my own law:  start {e2[0]:.1f} deg  ->  final {e2[-1]:.3f} deg")

## More

- Landing page: https://nscheuer.github.io/Generalized_ADCS/ssc26/
- Repository: https://github.com/nscheuer/Generalized_ADCS
- Every code block on the poster is executed in CI by
  `papers/SSC26_poster/verify_snippets.py`, so printed code cannot go stale.